# Customer Segmentation using RFM Analysis

## Project Overview

This notebook performs customer segmentation using the RFM (Recency, Frequency, Monetary) framework on the Online Retail II dataset.

The objective is to identify valuable customer groups based on purchasing behavior and generate business insights that can improve customer retention, marketing campaigns, and revenue optimization.

**Tech Stack**
- Python
- Pandas
- NumPy
- PostgreSQL
- Power BI

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Step 1: Load the Cleaned Dataset
The cleaned dataset generated during the data cleaning phase is loaded for RFM analysis.

In [2]:
df = pd.read_csv(r'C:\Users\prasa\cleaned_retail.csv')

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 779425 entries, 0 to 779424
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      779425 non-null  int64  
 1   StockCode    779425 non-null  object 
 2   Description  779425 non-null  object 
 3   Quantity     779425 non-null  int64  
 4   InvoiceDate  779425 non-null  object 
 5   Price        779425 non-null  float64
 6   Customer ID  779425 non-null  int64  
 7   Country      779425 non-null  object 
 8   TotalPrice   779425 non-null  float64
dtypes: float64(2), int64(3), object(4)
memory usage: 53.5+ MB


## Step 2: Convert InvoiceDate to DateTime

The InvoiceDate column is converted from string format to datetime format to enable time-based calculations such as Recency.

In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 779425 entries, 0 to 779424
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      779425 non-null  int64         
 1   StockCode    779425 non-null  object        
 2   Description  779425 non-null  object        
 3   Quantity     779425 non-null  int64         
 4   InvoiceDate  779425 non-null  datetime64[ns]
 5   Price        779425 non-null  float64       
 6   Customer ID  779425 non-null  int64         
 7   Country      779425 non-null  object        
 8   TotalPrice   779425 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 53.5+ MB


## Step 3: Create Snapshot Date

In [7]:
df['InvoiceDate'].max()

Timestamp('2011-12-09 12:50:00')

In [8]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [9]:
snapshot_date

Timestamp('2011-12-10 12:50:00')

## Step 4: Calculate RFM Metrics

Group transactions by Customer ID to calculate:

- Recency (Last Purchase Date)
- Frequency (Unique Invoices)
- Monetary (Total Spending)

In [10]:
Rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': 'max',
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
})

In [11]:
Rfm.head()

,InvoiceDate,Invoice,TotalPrice
Customer ID,,,
12346,2011-01-18 10:01:00,12,77556.46
12347,2011-12-07 15:52:00,8,4921.53
12348,2011-09-25 13:13:00,5,2019.40
12349,2011-11-21 09:51:00,4,4428.69
12350,2011-02-02 16:01:00,1,334.40


In [12]:
Rfm.columns = ['LastPurchaseDate', 'Frequency', 'Monetary']

In [13]:
Rfm.head()

,LastPurchaseDate,Frequency,Monetary
Customer ID,,,
12346,2011-01-18 10:01:00,12,77556.46
12347,2011-12-07 15:52:00,8,4921.53
12348,2011-09-25 13:13:00,5,2019.40
12349,2011-11-21 09:51:00,4,4428.69
12350,2011-02-02 16:01:00,1,334.40


In [14]:
Rfm['Recency'] = (snapshot_date - Rfm['LastPurchaseDate']).dt.days

In [15]:
Rfm.head()

,LastPurchaseDate,Frequency,Monetary,Recency
Customer ID,,,,
12346,2011-01-18 10:01:00,12,77556.46,326
12347,2011-12-07 15:52:00,8,4921.53,2
12348,2011-09-25 13:13:00,5,2019.40,75
12349,2011-11-21 09:51:00,4,4428.69,19
12350,2011-02-02 16:01:00,1,334.40,310


In [16]:
Rfm = Rfm.drop(columns=['LastPurchaseDate'])

In [17]:
Rfm = Rfm[['Recency', 'Frequency', 'Monetary']]

In [18]:
Rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346,326,12,77556.46
12347,2,8,4921.53
12348,75,5,2019.40
12349,19,4,4428.69
12350,310,1,334.40


In [19]:
Rfm.shape

(5878, 3)

In [20]:
Rfm['R_Score'] = pd.qcut(
    Rfm['Recency'],
    q=5,
    labels=[5, 4, 3, 2, 1]
)

In [21]:
Rfm[['Recency', 'R_Score']].head(10)

,Recency,R_Score
Customer ID,,
12346,326,2
12347,2,5
12348,75,3
12349,19,5
12350,310,2
12351,375,2
12352,36,4
12353,204,2
12354,232,2


In [22]:
Rfm['R_Score'].value_counts().sort_index()

R_Score
5    1188
4    1176
3    1167
2    1172
1    1175
Name: count, dtype: int64

In [23]:
Rfm['Frequency'].describe()

count    5878.000000
mean        6.289384
std        13.009406
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       398.000000
Name: Frequency, dtype: float64

In [24]:
Rfm['Frequency'].value_counts().head(20)

Frequency
1     1623
2      944
3      664
4      486
5      360
6      280
7      222
8      172
9      153
11     131
10      98
12      96
13      79
15      51
14      49
16      43
17      41
18      32
20      31
19      26
Name: count, dtype: int64

In [25]:
Rfm['F_Score'] = pd.qcut(
    Rfm['Frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

In [26]:
Rfm[['Frequency', 'F_Score']].head(10)


,Frequency,F_Score
Customer ID,,
12346,12,5
12347,8,4
12348,5,4
12349,4,3
12350,1,1
12351,1,1
12352,10,5
12353,2,2
12354,1,1


In [27]:
Rfm['F_Score'].value_counts().sort_index()

F_Score
1    1176
2    1175
3    1176
4    1175
5    1176
Name: count, dtype: int64

In [28]:
Rfm['M_Score'] = pd.qcut(
    Rfm['Monetary'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

In [29]:
Rfm[['Monetary', 'M_Score']].head(10)

,Monetary,M_Score
Customer ID,,
12346,77556.46,5
12347,4921.53,5
12348,2019.40,4
12349,4428.69,5
12350,334.40,2
12351,300.93,2
12352,2849.84,4
12353,406.76,2
12354,1079.40,3


In [30]:
Rfm['M_Score'].value_counts().sort_index()

M_Score
1    1176
2    1175
3    1176
4    1175
5    1176
Name: count, dtype: int64

In [31]:
Rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5878 entries, 12346 to 18287
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Recency    5878 non-null   int64   
 1   Frequency  5878 non-null   int64   
 2   Monetary   5878 non-null   float64 
 3   R_Score    5878 non-null   category
 4   F_Score    5878 non-null   category
 5   M_Score    5878 non-null   category
dtypes: category(3), float64(1), int64(2)
memory usage: 201.5 KB


In [32]:
Rfm['RFM_Score'] = (
    Rfm['R_Score'].astype(str) +
    Rfm['F_Score'].astype(str) +
    Rfm['M_Score'].astype(str)
)

In [33]:
Rfm[['R_Score','F_Score','M_Score','RFM_Score']].head(10)

,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,
12346,2,5,5,255
12347,5,4,5,545
12348,3,4,4,344
12349,5,3,5,535
12350,2,1,2,212
12351,2,1,2,212
12352,4,5,4,454
12353,2,2,2,222
12354,2,1,3,213


In [34]:
Rfm['R_Score'] = Rfm['R_Score'].astype(int)
Rfm['F_Score'] = Rfm['F_Score'].astype(int)
Rfm['M_Score'] = Rfm['M_Score'].astype(int)

In [35]:
Rfm[['R_Score','F_Score','M_Score','RFM_Score']].head(10)

,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,
12346,2,5,5,255
12347,5,4,5,545
12348,3,4,4,344
12349,5,3,5,535
12350,2,1,2,212
12351,2,1,2,212
12352,4,5,4,454
12353,2,2,2,222
12354,2,1,3,213


In [36]:
Rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5878 entries, 12346 to 18287
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Recency    5878 non-null   int64  
 1   Frequency  5878 non-null   int64  
 2   Monetary   5878 non-null   float64
 3   R_Score    5878 non-null   int64  
 4   F_Score    5878 non-null   int64  
 5   M_Score    5878 non-null   int64  
 6   RFM_Score  5878 non-null   object 
dtypes: float64(1), int64(5), object(1)
memory usage: 367.4+ KB


In [37]:
champions = (
    (Rfm['R_Score'] == 5) &
    (Rfm['F_Score'] >= 4) &
    (Rfm['M_Score'] >= 4)
)

In [38]:
champions.sum()

np.int64(741)

In [39]:
loyal_customers = (
    (Rfm['R_Score'] >= 3) & 
    (Rfm['F_Score'] >= 4) & 
    (Rfm['M_Score'] >= 3)
)

In [40]:
loyal_customers.sum()

np.int64(1947)

In [41]:
potential_loyalists = (
    (Rfm['R_Score'] >= 4) & 
    (Rfm['F_Score'] >= 2) & 
    (Rfm['M_Score'] >= 2)
)

In [42]:
potential_loyalists.sum()

np.int64(2093)

In [43]:
new_customers = (
    (Rfm['R_Score'] == 5) & 
    (Rfm['F_Score'] == 1)
)

In [44]:
new_customers.sum()

np.int64(54)

In [45]:
at_risk = (
    (Rfm['R_Score'] <= 2) & 
    (Rfm['F_Score'] >= 4) & 
    (Rfm['M_Score'] >= 3)
)

In [46]:
at_risk.sum()

np.int64(327)

In [47]:
hibernating = (
    (Rfm['R_Score'] <= 2) & 
    (Rfm['F_Score'] <= 2)
)

In [49]:
hibernating.sum()

np.int64(1523)

In [51]:
conditions = [
    champions,
    loyal_customers,
    potential_loyalists,
    new_customers,
    at_risk,
    hibernating
]

choices = [
    'Champions',
    'Loyal Customers',
    'Potential Loyalists',
    'New Customers',
    'At Risk',
    'Hibernating'
]

Rfm['Segment'] = np.select(
    conditions,
    choices,
    default='Others'
)

In [53]:
Rfm['Segment'].value_counts()

Segment
Hibernating            1523
Others                 1394
Loyal Customers        1206
Champions               741
Potential Loyalists     633
At Risk                 327
New Customers            54
Name: count, dtype: int64

In [55]:
Rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
Customer ID,,,,,,,,
12346,326,12,77556.46,2,5,5,255,At Risk
12347,2,8,4921.53,5,4,5,545,Champions
12348,75,5,2019.40,3,4,4,344,Loyal Customers
12349,19,4,4428.69,5,3,5,535,Potential Loyalists
12350,310,1,334.40,2,1,2,212,Hibernating


In [57]:
Rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2)

,Recency,Frequency,Monetary
Segment,,,
At Risk,340.69,7.79,3301.39
Champions,8.29,21.10,11909.57
Hibernating,459.28,1.25,429.70
Loyal Customers,62.30,9.72,4036.84
New Customers,10.50,1.00,359.75
Others,197.03,2.39,804.24
Potential Loyalists,25.35,2.80,1274.67


In [59]:
Rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5878 entries, 12346 to 18287
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Recency    5878 non-null   int64  
 1   Frequency  5878 non-null   int64  
 2   Monetary   5878 non-null   float64
 3   R_Score    5878 non-null   int64  
 4   F_Score    5878 non-null   int64  
 5   M_Score    5878 non-null   int64  
 6   RFM_Score  5878 non-null   object 
 7   Segment    5878 non-null   object 
dtypes: float64(1), int64(5), object(2)
memory usage: 413.3+ KB


In [61]:
Rfm = Rfm.reset_index()

In [63]:
Rfm.head()

,index,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,0,12346,326,12,77556.46,2,5,5,255,At Risk
1,1,12347,2,8,4921.53,5,4,5,545,Champions
2,2,12348,75,5,2019.40,3,4,4,344,Loyal Customers
3,3,12349,19,4,4428.69,5,3,5,535,Potential Loyalists
4,4,12350,310,1,334.40,2,1,2,212,Hibernating


In [65]:
Rfm.columns

Index(['index', 'Customer ID', 'Recency', 'Frequency', 'Monetary', 'R_Score',
       'F_Score', 'M_Score', 'RFM_Score', 'Segment'],
      dtype='object')

In [66]:
Rfm.index

RangeIndex(start=0, stop=5878, step=1)

In [67]:
Rfm = Rfm.drop(columns=['index'])

In [68]:
Rfm.columns

Index(['Customer ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score',
       'M_Score', 'RFM_Score', 'Segment'],
      dtype='object')

In [69]:
Rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346,326,12,77556.46,2,5,5,255,At Risk
1,12347,2,8,4921.53,5,4,5,545,Champions
2,12348,75,5,2019.40,3,4,4,344,Loyal Customers
3,12349,19,4,4428.69,5,3,5,535,Potential Loyalists
4,12350,310,1,334.40,2,1,2,212,Hibernating


In [70]:
Rfm.to_csv('rfm_scored.csv', index=False)